# Archived-result reproduction
Data/results are not uploaded to GitHub. See [download and preparation instructions](../DATA.md). Raw source data alone do not contain the derived alignment results used here. Set `RAFTUP_EXPERIMENT_ROOT` to your revision archive root. These cells recompute plots and evaluations from saved results, not model training.


# Downsampling sensitivity: three datasets and six ratios

从已保存的 18 次运行结果重绘，不训练、不重新配准。
先运行路径、数据读取和绘图设置三个公共 cell，然后可独立运行任意子图 cell。

每个数据集的 LAA、GPR、Coverage A、Coverage B 和运行时间分别绘图，共 15 张。
字体在公共设置中统一调整；不添加字母标注；导出时四周留白为零。
图片保存到 `outputs/04_downsampling/panels/`（PDF、SVG、600 dpi PNG 和预览 PNG）。

原始文件：
`<revision_archive>/all_exp/downsampling_sensitivity/01_downsampling_sensitivity_3_real_datasets.ipynb`


In [ ]:
from pathlib import Path
import os, sys
os.environ.setdefault('MPLCONFIGDIR', '/tmp/raftup_reviewer_figures_mpl')
# Input project root. Change only this path when moving the project.
ROOT = Path(os.environ.get('RAFTUP_EXPERIMENT_ROOT', './revision_data')).expanduser().resolve()
NOTEBOOK_DIR = Path.cwd()
RESULTS_DIR = Path(os.environ.get('RAFTUP_OUTPUT_ROOT', './outputs')).expanduser().resolve()
OUT = RESULTS_DIR / '04_downsampling'
OUT.mkdir(parents=True, exist_ok=True)
assert ROOT.is_dir(), f'Missing input project: {ROOT}'
print('Input project:', ROOT)
print('Outputs:', OUT)

import json, itertools
import numpy as np
import pandas as pd
from IPython.display import display
ARCHIVE_OUT = ROOT / 'all_exp/downsampling_sensitivity/results_single_seed_v1'
SUPPLEMENT_OUT = ROOT / 'all_exp/downsampling_sensitivity/results_three_real_no_cap_v2'
REPORT_OUT = OUT
SELECTED_CASES = ['dlpfc_0708', 'merfish_0409', 'deep_starmap_13_14']
RATIOS = [0.02, 0.04, 0.05, 0.067, 0.08, 0.10]
FIXED_SEED = 42
REFERENCE_RATIO = 0.05
SUPPLEMENT_CASE = 'deep_starmap_13_14'
SUPPLEMENT_RATIO = 0.10
REUSE_EXISTING_RESULTS = False  # Select the saved supplemental 10% result; does not launch runs.


In [ ]:
# Reload results without repeating alignment.
records=[]
result_folders={}
for name, ratio in itertools.product(SELECTED_CASES, RATIOS):
    use_new = not REUSE_EXISTING_RESULTS and name == SUPPLEMENT_CASE and np.isclose(ratio, SUPPLEMENT_RATIO)
    source = SUPPLEMENT_OUT if use_new else ARCHIVE_OUT
    folder=source/"runs"/name/f"ratio_{ratio:.4f}"/f"seed_{FIXED_SEED}"
    result_folders[(name, ratio)]=folder
    path=folder/"result.json"
    if path.exists():
        row=json.loads(path.read_text())
        row["result_source"]=str(path)
        row["source_kind"]="supplement_no_cap" if use_new else "archived_800_cap"
        records.append(row)
    else:
        raise FileNotFoundError(f"Missing archived result: {path}")
df=pd.DataFrame(records)
df=df[df.case.isin(SELECTED_CASES)].copy()
assert len(df) == 18, "Expected all 18 saved runs"
assert df.seed.eq(FIXED_SEED).all(), "Use the single-seed output directory"
assert not df.duplicated(["case","ratio"]).any()
metrics=["LAA","GPR","coverage_A","coverage_B","alignment_seconds","sampling_seconds","step1_seconds","step2_seconds","ratio_a","ratio_b"]
for col in metrics:
    if col not in df: df[col]=np.nan
# Include not-yet-run settings so interrupted sweeps remain visible.
summary=pd.DataFrame(itertools.product(SELECTED_CASES,RATIOS),columns=["case","ratio"]).merge(df,on=["case","ratio"],how="left")
summary["status"]=summary.status.fillna("not_run")
summary.to_csv(REPORT_OUT/"single_run_summary.csv",index=False)

def jaccard(a,b):
    a,b=set(a),set(b);return len(a&b)/len(a|b) if a|b else np.nan

def load_matches(name,ratio):
    path=result_folders[(name,ratio)]/"hard_matches.npz"
    with np.load(path) as f:return {k:f[k] for k in f.files}

comparisons=[]
for name in SELECTED_CASES:
    comparisons_to_make=[("vs_5pct",REFERENCE_RATIO,r) for r in RATIOS if not np.isclose(r,REFERENCE_RATIO)]
    comparisons_to_make += [("adjacent_ratios",r1,r2) for r1,r2 in zip(sorted(RATIOS)[:-1],sorted(RATIOS)[1:])]
    for kind,r1,r2 in comparisons_to_make:
        left=summary[(summary.case==name)&np.isclose(summary.ratio,r1)].iloc[0]
        right=summary[(summary.case==name)&np.isclose(summary.ratio,r2)].iloc[0]
        row=dict(case=name,comparison=kind,ratio_from=r1,ratio_to=r2,status_from=left.status,status_to=right.status)
        if left.status=="ok" and right.status=="ok":
            a,b=load_matches(name,r1),load_matches(name,r2)
            p,q=a["prediction"],b["prediction"];pa=p>=0;pb=q>=0
            same=(p==q)&pa&pb;union=int(pa.sum()+pb.sum()-same.sum());common=pa&pb
            row.update(hard_match_jaccard=float(same.sum()/union) if union else np.nan,
                common_covered_agreement=float(same[common].mean()) if common.any() else np.nan,
                coverage_A_jaccard=jaccard(np.flatnonzero(pa),np.flatnonzero(pb)),
                coverage_B_jaccard=jaccard(np.flatnonzero(a["coverage_B"]),np.flatnonzero(b["coverage_B"])),
                sample_A_jaccard=jaccard(a["sample_A"],b["sample_A"]),sample_B_jaccard=jaccard(a["sample_B"],b["sample_B"]),
                LAA_delta=right.LAA-left.LAA,GPR_delta=right.GPR-left.GPR,
                runtime_ratio=right.alignment_seconds/left.alignment_seconds if left.alignment_seconds>0 else np.nan)
        comparisons.append(row)
changes=pd.DataFrame(comparisons)
for col in ["hard_match_jaccard","common_covered_agreement","LAA_delta","GPR_delta","runtime_ratio"]:
    if col not in changes:changes[col]=np.nan
changes.to_csv(REPORT_OUT/"between_ratio_consistency.csv",index=False)
display(summary[["case","ratio","status",*metrics]])
display(changes)


In [ ]:
import matplotlib.pyplot as plt
from IPython.display import Image as DisplayImage
FIG = REPORT_OUT / "panels"
FIG.mkdir(exist_ok=True)
plt.rcParams.update({
    "font.family": "Arial", "font.size": 18,
    "axes.titlesize": 20, "axes.labelsize": 18,
    "xtick.labelsize": 16, "ytick.labelsize": 16,
    "legend.fontsize": 14, "axes.linewidth": 1.1,
    "pdf.fonttype": 42, "ps.fonttype": 42, "svg.fonttype": "none",
    "savefig.facecolor": "white",
})
colors = plt.get_cmap("tab10").colors
CASE_LABELS = {"dlpfc_0708": "DLPFC_0708", "merfish_0409": "MERFISH_0409", "deep_starmap_13_14": "Deep-STARmap_1314"}
AUTO_BASELINE_RATIOS = {"dlpfc_0708": 0.05, "merfish_0409": 0.05, "deep_starmap_13_14": 0.067}

def plot_metric(name, metric, title):
    sub = summary[summary.case == name].sort_values("ratio").copy()
    values = sub[metric].where(sub.status == "ok")
    fig, ax = plt.subplots(figsize=(6.4, 4.8), layout="constrained")
    ax.plot(sub.ratio * 100, values, marker="o", markersize=7,
            linewidth=2, color=colors[SELECTED_CASES.index(name)])
    ax.axvline(AUTO_BASELINE_RATIOS[name] * 100, color="#6b21a8",
               linestyle="--", linewidth=1.5, alpha=.85,
               zorder=0, label="Default ratio")
    ax.set_title(CASE_LABELS[name], pad=12)
    ax.set_xlabel("Downsampling ratio (%)")
    ax.set_ylabel(title)
    ax.set_xticks(np.array(RATIOS) * 100, [f"{r * 100:g}" for r in RATIOS])
    if metric != "alignment_seconds":
        ax.set_ylim(-.03, 1.03)
    ax.grid(alpha=.2)
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(loc="best", framealpha=.85)
    return fig, ax

def savefig(fig, name):
    for ext in ["png", "pdf", "svg"]:
        fig.savefig(FIG / f"{name}.{ext}", dpi=600,
                    bbox_inches="tight", pad_inches=0)
    preview = FIG / f"{name}_preview.png"
    fig.savefig(preview, dpi=180, bbox_inches="tight", pad_inches=0)
    plt.close(fig)
    display(DisplayImage(filename=str(preview)))
    print(FIG / f"{name}.pdf")


## dlpfc_0708 — LAA


In [ ]:
fig, ax = plot_metric("dlpfc_0708", "LAA", "LAA")
# Customize this subplot here before saving.
savefig(fig, "dlpfc_0708_LAA")


## dlpfc_0708 — GPR


In [ ]:
fig, ax = plot_metric("dlpfc_0708", "GPR", "GPR")
# Customize this subplot here before saving.
savefig(fig, "dlpfc_0708_GPR")


## dlpfc_0708 — Coverage A


In [ ]:
fig, ax = plot_metric("dlpfc_0708", "coverage_A", "Coverage A")
# Customize this subplot here before saving.
savefig(fig, "dlpfc_0708_coverage_A")


## dlpfc_0708 — Coverage B


In [ ]:
fig, ax = plot_metric("dlpfc_0708", "coverage_B", "Coverage B")
# Customize this subplot here before saving.
savefig(fig, "dlpfc_0708_coverage_B")


## dlpfc_0708 — Runtime (s)


In [ ]:
fig, ax = plot_metric("dlpfc_0708", "alignment_seconds", "Runtime (s)")
# Customize this subplot here before saving.
savefig(fig, "dlpfc_0708_alignment_seconds")


## merfish_0409 — LAA


In [ ]:
fig, ax = plot_metric("merfish_0409", "LAA", "LAA")
# Customize this subplot here before saving.
savefig(fig, "merfish_0409_LAA")


## merfish_0409 — GPR


In [ ]:
fig, ax = plot_metric("merfish_0409", "GPR", "GPR")
# Customize this subplot here before saving.
savefig(fig, "merfish_0409_GPR")


## merfish_0409 — Coverage A


In [ ]:
fig, ax = plot_metric("merfish_0409", "coverage_A", "Coverage A")
# Customize this subplot here before saving.
savefig(fig, "merfish_0409_coverage_A")


## merfish_0409 — Coverage B


In [ ]:
fig, ax = plot_metric("merfish_0409", "coverage_B", "Coverage B")
# Customize this subplot here before saving.
savefig(fig, "merfish_0409_coverage_B")


## merfish_0409 — Runtime (s)


In [ ]:
fig, ax = plot_metric("merfish_0409", "alignment_seconds", "Runtime (s)")
# Customize this subplot here before saving.
savefig(fig, "merfish_0409_alignment_seconds")


## deep_starmap_13_14 — LAA


In [ ]:
fig, ax = plot_metric("deep_starmap_13_14", "LAA", "LAA")
# Customize this subplot here before saving.
savefig(fig, "deep_starmap_13_14_LAA")


## deep_starmap_13_14 — GPR


In [ ]:
fig, ax = plot_metric("deep_starmap_13_14", "GPR", "GPR")
# Customize this subplot here before saving.
savefig(fig, "deep_starmap_13_14_GPR")


## deep_starmap_13_14 — Coverage A


In [ ]:
fig, ax = plot_metric("deep_starmap_13_14", "coverage_A", "Coverage A")
# Customize this subplot here before saving.
savefig(fig, "deep_starmap_13_14_coverage_A")


## deep_starmap_13_14 — Coverage B


In [ ]:
fig, ax = plot_metric("deep_starmap_13_14", "coverage_B", "Coverage B")
# Customize this subplot here before saving.
savefig(fig, "deep_starmap_13_14_coverage_B")


## deep_starmap_13_14 — Runtime (s)


In [ ]:
fig, ax = plot_metric("deep_starmap_13_14", "alignment_seconds", "Runtime (s)")
# Customize this subplot here before saving.
savefig(fig, "deep_starmap_13_14_alignment_seconds")
